In [8]:
import torch
from PIL import Image
from transformers import CLIPProcessor, CLIPModel
import os
import json

In [ ]:
# Load CLIP model and processor
model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

# Sample wardrobe images (replace with actual file paths)
WARDROBE_DIR = os.path.join(os.getcwd(), "wardrobe")
wardrobe_images = {
    os.path.splitext(os.path.basename(img))[0]: img
    for img in glob.glob(os.path.join(WARDROBE_DIR, "*.jpg"))
}

# Define a mood input
mood_text = "chill and relaxed"

def get_best_matching_outfit(mood, wardrobe):
    """Finds the best outfit match for the given mood using CLIP embeddings."""
    # Process text
    text_inputs = processor(text=[mood], return_tensors="pt", padding=True)
    text_embedding = model.get_text_features(**text_inputs).detach()
    
    best_match = None
    best_score = float('-inf')
    
    for item, img_path in wardrobe.items():
        image = Image.open(img_path).convert("RGB")
        image_inputs = processor(images=image, return_tensors="pt")
        image_embedding = model.get_image_features(**image_inputs).detach()
        
        # Compute cosine similarity
        similarity = torch.nn.functional.cosine_similarity(text_embedding, image_embedding)
        
        if similarity > best_score:
            best_match = item
            best_score = similarity
    
    return best_match

# Get outfit suggestion
best_outfit = get_best_matching_outfit(mood_text, wardrobe_images)
print(f"Recommended outfit for '{mood_text}': {best_outfit}")

Recommended outfit for 'red': red_jacket


In [13]:
import requests
from bs4 import BeautifulSoup

def scrape_fashion_trends():
    url = "https://www.vogue.com/fashion/trends"  # Example source
    headers = {"User-Agent": "Mozilla/5.0"}

    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.text, "html.parser")

    trends = []
    print(soup.find_all("article", limit=10))
    for article in soup.find_all("article", limit=10):
        title = article.find("h2").text.strip() if article.find("h2") else "No Title"
        summary = article.find("p").text.strip() if article.find("p") else "No Summary"
        trends.append({"title": title, "summary": summary})

    return trends

# Fetch and print scraped fashion trends
fashion_trends = scrape_fashion_trends()
print("Trends:")
for trend in fashion_trends:
    print(f"Trend: {trend['title']}\nSummary: {trend['summary']}\n")

[]
Trends:


In [1]:
import db.db_client as db
from models.constants import ITEM_TYPE

db.init_db()
# db.add_item(name="red hoodie", item_type=ITEM_TYPE.OUTER, description="plain bright red cotton hoodie", tags=["casual", "winter"])
print(db.list_items(only_available=True))

[(1, 'red hoodie', 'OUTER', 'plain bright red cotton hoodie', 'casual,winter', '', 1, None, None)]


In [2]:
db.list_items()

[(1,
  'red hoodie',
  'OUTER',
  'plain bright red cotton hoodie',
  'casual,winter',
  '',
  1,
  None,
  None)]